# Mapping the cosmic web — full pipeline

One notebook for the whole arc we built, end to end:

1. **A field generator** — Lagrangian perturbation theory, 1st order (Zel'dovich) or 2nd order (2LPT).
2. **Map the web** — classify every voxel void / sheet / filament / knot with the tidal-tensor (T-web) method.
3. **The 2LPT upgrade** — show what second order buys over Zel'dovich.
4. **Infer the web from sparse tracers** — a 2D U-Net that recovers the web from a shot-noisy galaxy-like sampling, plus how it degrades as the survey thins.
5. **Go 3D** — the same idea with a 3D U-Net on sub-cubes (uses a GPU automatically if present).

Everything here is pure NumPy + PyTorch, runs locally, and is a toy stand-in for real N-body data — which is the next step after this. Run top to bottom.

## 0. Setup

In [ ]:
# run once
!pip install torch

In [ ]:
import os, time, numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from numpy.fft import fftn, ifftn, fftfreq
torch.manual_seed(1); np.random.seed(1); torch.set_num_threads(os.cpu_count() or 1)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- config ----
L          = 250.0     # box size, Mpc/h  (shared everywhere)
sigma_lin  = 2.0       # linear-density rms: nonlinearity / 2LPT strength
R_smooth   = 3.0       # Gaussian smoothing (Mpc/h) before T-web
lam_th     = 0.3       # tidal-tensor eigenvalue threshold
N_map      = 128       # grid for the map + 2LPT comparison (visual quality)
N_unet     = 64        # grid for the U-Net data (kept small to train fast)
nbar_base  = 2.0       # mean tracers/voxel for the headline U-Net runs
nbar_sweep = [2.0, 0.5, 0.25]   # sparsity sweep
f2d, epochs2d = 16, 12
patch, stride, f3d, epochs3d = 32, 16, 12, 12

CLASS_NAMES  = ['void','sheet','filament','knot']
CLASS_COLORS = ['#08306b','#6baed6','#fd8d3c','#cb181d']
CMAP = ListedColormap(CLASS_COLORS); NORM = BoundaryNorm([-.5,.5,1.5,2.5,3.5],4)
print("torch", torch.__version__, "| device:", device, "| threads", torch.get_num_threads())

## 1. The data layer: field, web labels, tracers, patches

`lpt_delta(order=1)` is Zel'dovich; `order=2` adds the 2LPT second-order displacement, sourced by the same Hessian of the potential the T-web uses. `tweb` classifies the web; `make_cube` builds a field plus a sparse Poisson-tracer view of it; `patches` cuts a cube into 3D sub-cubes.

In [ ]:
def Pk_bbks(k, ns=0.96, G=0.2):
    k=np.asarray(k,float); q=np.where(k>0,k/G,1e-10)
    T=(np.log(1+2.34*q)/(2.34*q))*(1+3.89*q+(16.1*q)**2+(5.46*q)**3+(6.71*q)**4)**(-0.25)
    return k**ns*T**2

def cic(pos,N,L):
    g=np.zeros((N,N,N)); s=(pos%L)/(L/N); i=np.floor(s).astype(int)%N; fr=s-np.floor(s)
    for dx in (0,1):
        for dy in (0,1):
            for dz in (0,1):
                wx=fr[:,0] if dx else 1-fr[:,0]; wy=fr[:,1] if dy else 1-fr[:,1]; wz=fr[:,2] if dz else 1-fr[:,2]
                np.add.at(g,((i[:,0]+dx)%N,(i[:,1]+dy)%N,(i[:,2]+dz)%N),wx*wy*wz)
    return g

def lpt_delta(N,L,sigma_lin,seed,order=2):
    kax=fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    Kv=[KX,KY,KZ]; k2=KX**2+KY**2+KZ**2; k2[0,0,0]=1.0
    rng=np.random.default_rng(seed)
    dk=fftn(rng.standard_normal((N,N,N)))*np.sqrt(Pk_bbks(np.sqrt(k2))); dk[0,0,0]=0
    d=ifftn(dk).real; d*=sigma_lin/d.std(); dk=fftn(d); src=dk.copy()
    if order==2:                                              # 2LPT second-order source
        pij=lambda a,b: ifftn(Kv[a]*Kv[b]/k2*dk).real
        pxx,pyy,pzz=pij(0,0),pij(1,1),pij(2,2); pxy,pxz,pyz=pij(0,1),pij(0,2),pij(1,2)
        S2=pxx*pyy+pxx*pzz+pyy*pzz-pxy**2-pxz**2-pyz**2
        src=dk+(3.0/7.0)*fftn(S2)
    disp=np.array([ifftn(1j*Kv[i]*src/k2).real for i in range(3)])
    q=np.arange(N)*(L/N); QX,QY,QZ=np.meshgrid(q,q,q,indexing='ij'); Q=[QX,QY,QZ]
    pos=np.stack([(Q[i]+disp[i]).ravel() for i in range(3)],axis=1)
    rho=cic(pos,N,L); return rho/rho.mean()-1.0

def smooth(d,L,R):
    N=d.shape[0]; kax=fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    return ifftn(fftn(d)*np.exp(-0.5*(KX**2+KY**2+KZ**2)*R**2)).real

def tweb(d,L,lam=0.3):
    N=d.shape[0]; kax=fftfreq(N,d=L/N)*2*np.pi; KX,KY,KZ=np.meshgrid(kax,kax,kax,indexing='ij')
    Kv=[KX,KY,KZ]; k2=KX**2+KY**2+KZ**2; k2[0,0,0]=1.0; dk=fftn(d); T=np.zeros((N,N,N,3,3))
    for i in range(3):
        for j in range(i,3):
            t=ifftn((Kv[i]*Kv[j]/k2)*dk).real; T[...,i,j]=t
            if i!=j: T[...,j,i]=t
    return np.sum(np.linalg.eigvalsh(T)>lam,-1).astype(np.int64)

def make_cube(N,L,seed,nbar,sigma_lin,order=2):
    d=lpt_delta(N,L,sigma_lin,seed,order); lab=tweb(smooth(d,L,R_smooth),L,lam_th)
    rng=np.random.default_rng(seed+999)
    counts=rng.poisson(nbar*np.clip(1+d,0,None)); dsp=counts/counts.mean()-1.0   # sparse tracers
    return d, dsp.astype(np.float32), lab

def to_slices(dsp,lab):
    X,Y=[],[]
    for ax in range(3):
        ds=np.moveaxis(dsp,ax,0); ls=np.moveaxis(lab,ax,0)
        for k in range(ds.shape[0]): X.append(ds[k]); Y.append(ls[k])
    return np.array(X),np.array(Y)

def patches(vol,lab,p,stride):
    N=vol.shape[0]; X,Y=[],[]
    for i in range(0,N-p+1,stride):
        for j in range(0,N-p+1,stride):
            for k in range(0,N-p+1,stride):
                X.append(vol[i:i+p,j:j+p,k:k+p]); Y.append(lab[i:i+p,j:j+p,k:k+p])
    return np.array(X),np.array(Y)

def skew(d): x=d-d.mean(); return (x**3).mean()/x.std()**3
def fracs(lab): f=np.bincount(lab.ravel(),minlength=4); return (100*f/f.sum()).round(1)
def per_class_f1(pred,true,nc=4):
    o=[]
    for c in range(nc):
        tp=((pred==c)&(true==c)).sum(); fp=((pred==c)&(true!=c)).sum(); fn=((pred!=c)&(true==c)).sum()
        o.append(2*tp/(2*tp+fp+fn+1e-9))
    return o
print("data layer ready")

## 2. Map the cosmic web

Generate a 2LPT field and classify it. Left: density; right: the four-class web.

In [ ]:
delta, _, labels = make_cube(N_map, L, seed=42, nbar=nbar_base, sigma_lin=sigma_lin, order=2)
print("web fractions %:", dict(zip(CLASS_NAMES, fracs(labels))))
z = N_map//2
fig,ax=plt.subplots(1,2,figsize=(13,6))
im0=ax[0].imshow(np.log10(1+delta[z].T+1e-3),origin='lower',cmap='inferno',extent=[0,L,0,L],vmin=-1.5,vmax=1.0)
ax[0].set_title(r'density  $\log_{10}(1+\delta)$'); ax[0].set_xlabel('Mpc/h'); ax[0].set_ylabel('Mpc/h')
plt.colorbar(im0,ax=ax[0],fraction=0.046)
im1=ax[1].imshow(labels[z].T,origin='lower',cmap=CMAP,norm=NORM,extent=[0,L,0,L])
ax[1].set_title('cosmic-web class'); ax[1].set_xlabel('Mpc/h')
cb=plt.colorbar(im1,ax=ax[1],ticks=[0,1,2,3],fraction=0.046); cb.ax.set_yticklabels(CLASS_NAMES)
plt.tight_layout(); plt.show()

## 3. The 2LPT upgrade

Same linear field, two orders. The web maps look alike at large scales; the **density PDF** is where 2LPT shows its hand — emptier voids, a heavier knot tail, higher skewness.

In [ ]:
delta_za = lpt_delta(N_map, L, sigma_lin, seed=42, order=1)
labels_za = tweb(smooth(delta_za,L,R_smooth),L,lam_th)
print(f"Zel'dovich : skew {skew(smooth(delta_za,L,2.0)):.2f}   fractions % {fracs(labels_za)}")
print(f"2LPT       : skew {skew(smooth(delta,L,2.0)):.2f}   fractions % {fracs(labels)}")

fig=plt.figure(figsize=(15,10))
def dens(ax,d,t): ax.imshow(np.log10(1+d[z].T+1e-3),origin='lower',cmap='inferno',vmin=-1.5,vmax=1.0,extent=[0,L,0,L]); ax.set_title(t); ax.set_xlabel('Mpc/h')
def web(ax,l,t): ax.imshow(l[z].T,origin='lower',cmap=CMAP,norm=NORM,extent=[0,L,0,L]); ax.set_title(t); ax.set_xlabel('Mpc/h')
dens(fig.add_subplot(2,3,1),delta_za,"Zel'dovich  density"); dens(fig.add_subplot(2,3,2),delta,"2LPT  density")
a3=fig.add_subplot(2,3,3); b=np.linspace(-1.2,1.5,60)
a3.hist(np.log10(1+delta_za.ravel()+1e-3),bins=b,histtype='step',color='#377eb8',lw=2,label="Zel'dovich",density=True)
a3.hist(np.log10(1+delta.ravel()+1e-3),bins=b,histtype='step',color='#e41a1c',lw=2,label="2LPT",density=True)
a3.set_yscale('log'); a3.set_xlabel(r'$\log_{10}(1+\delta)$'); a3.set_title('density PDF'); a3.legend()
web(fig.add_subplot(2,3,4),labels_za,"Zel'dovich  web"); web(fig.add_subplot(2,3,5),labels,"2LPT  web")
a6=fig.add_subplot(2,3,6); im=a6.imshow(labels[z].T,cmap=CMAP,norm=NORM); a6.set_visible(False)
cb=fig.colorbar(im,ax=a6,ticks=[0,1,2,3],fraction=0.3); cb.ax.set_yticklabels(CLASS_NAMES)
plt.tight_layout(); plt.show()

## 4. Infer the web from sparse tracers — 2D U-Net

Real surveys see a sparse, shot-noisy sampling, not the full field. Train a 2D U-Net (on slices) to recover the web class from that sparse view, then push the tracer density `nbar` down to see what breaks first.

In [ ]:
def double_conv2d(ci,co):
    return nn.Sequential(nn.Conv2d(ci,co,3,padding=1),nn.BatchNorm2d(co),nn.ReLU(True),
                         nn.Conv2d(co,co,3,padding=1),nn.BatchNorm2d(co),nn.ReLU(True))
class UNet2D(nn.Module):
    def __init__(self,f=16,nc=4):
        super().__init__(); self.e1=double_conv2d(1,f); self.e2=double_conv2d(f,2*f); self.b=double_conv2d(2*f,4*f)
        self.u2=nn.ConvTranspose2d(4*f,2*f,2,2); self.d2=double_conv2d(4*f,2*f)
        self.u1=nn.ConvTranspose2d(2*f,f,2,2); self.d1=double_conv2d(2*f,f); self.o=nn.Conv2d(f,nc,1); self.p=nn.MaxPool2d(2)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.p(e1)); b=self.b(self.p(e2))
        d2=self.d2(torch.cat([self.u2(b),e2],1)); d1=self.d1(torch.cat([self.u1(d2),e1],1)); return self.o(d1)

def train_eval_2d(nbar, order=2, N=N_unet, epochs=epochs2d, f=f2d):
    _,dtr,ltr=make_cube(N,L,1,nbar,sigma_lin,order); _,dva,lva=make_cube(N,L,2,nbar,sigma_lin,order)
    Xtr,Ytr=to_slices(dtr,ltr); Xva,Yva=to_slices(dva,lva)
    mu,sd=Xtr.mean(),Xtr.std()
    Xt=torch.tensor(((Xtr-mu)/sd)[:,None]).to(device); Yt=torch.tensor(Ytr).to(device)
    Xv=torch.tensor(((Xva-mu)/sd)[:,None]).to(device); Yv=torch.tensor(Yva).to(device)
    fr=np.bincount(Ytr.ravel(),minlength=4); w=(1.0/(fr+1)); w=(w/w.sum()*4).astype(np.float32)
    net=UNet2D(f).to(device); opt=torch.optim.Adam(net.parameters(),1e-3); crit=nn.CrossEntropyLoss(weight=torch.tensor(w).to(device))
    bs,n=16,len(Xt)
    for ep in range(epochs):
        net.train(); perm=torch.randperm(n)
        for i in range(0,n,bs):
            idx=perm[i:i+bs]; opt.zero_grad(); crit(net(Xt[idx]),Yt[idx]).backward(); opt.step()
    net.eval()
    with torch.no_grad(): pv=net(Xv).argmax(1).cpu().numpy()
    acc=(pv==Yva).mean()
    return acc, per_class_f1(pv,Yva), (Xva, Yva, pv)

t0=time.time()
acc2d, f1_2d, sample = train_eval_2d(nbar_base)
print(f"2D U-Net @ nbar={nbar_base}: valAcc {acc2d:.3f}  F1 "
      + "  ".join(f"{c} {v:.2f}" for c,v in zip(CLASS_NAMES,f1_2d)) + f"   ({time.time()-t0:.0f}s)")

In [ ]:
# prediction triptych: sparse input | true web | U-Net prediction
Xva,Yva,pv = sample; s=len(Xva)//2
fig,ax=plt.subplots(1,3,figsize=(15,5))
ax[0].imshow(Xva[s].T,origin='lower',cmap='inferno'); ax[0].set_title(f'sparse tracers (nbar={nbar_base})')
ax[1].imshow(Yva[s].T,origin='lower',cmap=CMAP,norm=NORM); ax[1].set_title('true web')
im=ax[2].imshow(pv[s].T,origin='lower',cmap=CMAP,norm=NORM); ax[2].set_title('2D U-Net prediction')
cb=plt.colorbar(im,ax=ax[2],ticks=[0,1,2,3],fraction=.046); cb.ax.set_yticklabels(CLASS_NAMES)
plt.tight_layout(); plt.show()

In [ ]:
# sparsity sweep
sweep={nbar_base:(acc2d,f1_2d)}
for nb in [x for x in nbar_sweep if x!=nbar_base]:
    t0=time.time(); a,F,_=train_eval_2d(nb); sweep[nb]=(a,F)
    print(f"nbar {nb:4.2f}: valAcc {a:.3f}  F1 "+"  ".join(f"{c} {v:.2f}" for c,v in zip(CLASS_NAMES,F))+f"   ({time.time()-t0:.0f}s)")
nbars=sorted(sweep,reverse=True); F=np.array([sweep[nb][1] for nb in nbars])
fig,ax=plt.subplots(figsize=(7,5))
for c in range(4): ax.plot(nbars,F[:,c],'o-',color=CLASS_COLORS[c],label=CLASS_NAMES[c])
ax.plot(nbars,[sweep[nb][0] for nb in nbars],'k--',label='overall acc')
ax.set_xscale('log'); ax.invert_xaxis(); ax.set_xlabel('mean tracers per voxel'); ax.set_ylabel('F1 / accuracy')
ax.set_title('Recovery vs survey sparsity'); ax.legend(); plt.tight_layout(); plt.show()

## 5. Go 3D — 3D U-Net on sub-cubes

The web is 3D. Same network with 3D convolutions, trained on overlapping sub-cubes instead of slices. Runs on the GPU automatically if one is present.

In [ ]:
def double_conv3d(ci,co):
    return nn.Sequential(nn.Conv3d(ci,co,3,padding=1),nn.BatchNorm3d(co),nn.ReLU(True),
                         nn.Conv3d(co,co,3,padding=1),nn.BatchNorm3d(co),nn.ReLU(True))
class UNet3D(nn.Module):
    def __init__(self,f=12,nc=4):
        super().__init__(); self.e1=double_conv3d(1,f); self.e2=double_conv3d(f,2*f); self.b=double_conv3d(2*f,4*f)
        self.u2=nn.ConvTranspose3d(4*f,2*f,2,2); self.d2=double_conv3d(4*f,2*f)
        self.u1=nn.ConvTranspose3d(2*f,f,2,2); self.d1=double_conv3d(2*f,f); self.o=nn.Conv3d(f,nc,1); self.p=nn.MaxPool3d(2)
    def forward(self,x):
        e1=self.e1(x); e2=self.e2(self.p(e1)); b=self.b(self.p(e2))
        d2=self.d2(torch.cat([self.u2(b),e2],1)); d1=self.d1(torch.cat([self.u1(d2),e1],1)); return self.o(d1)

_,dtr,ltr=make_cube(N_unet,L,1,nbar_base,sigma_lin,2); _,dva,lva=make_cube(N_unet,L,2,nbar_base,sigma_lin,2)
Xtr,Ytr=patches(dtr,ltr,patch,stride); Xva,Yva=patches(dva,lva,patch,stride)
mu,sd=Xtr.mean(),Xtr.std()
Xt=torch.tensor(((Xtr-mu)/sd)[:,None]).to(device); Yt=torch.tensor(Ytr).to(device)
Xv=torch.tensor(((Xva-mu)/sd)[:,None]).to(device); Yv=torch.tensor(Yva).to(device)
fr=np.bincount(Ytr.ravel(),minlength=4); w=(1.0/(fr+1)); w=(w/w.sum()*4).astype(np.float32)
net3=UNet3D(f3d).to(device); opt=torch.optim.Adam(net3.parameters(),1e-3); crit=nn.CrossEntropyLoss(weight=torch.tensor(w).to(device))
print(f"3D U-Net: {sum(p.numel() for p in net3.parameters())} params | {Xtr.shape[0]} train cubes of {tuple(Xtr.shape[1:])}")
bs,n=4,len(Xt); t0=time.time()
for ep in range(epochs3d):
    net3.train(); perm=torch.randperm(n)
    for i in range(0,n,bs):
        idx=perm[i:i+bs]; opt.zero_grad(); loss=crit(net3(Xt[idx]),Yt[idx]); loss.backward(); opt.step()
    if ep%4==3 or ep==epochs3d-1:
        net3.eval()
        with torch.no_grad(): pv=net3(Xv).argmax(1).cpu().numpy()
        F=per_class_f1(pv,Yva)
        print(f"epoch {ep+1:2d}  valAcc {(pv==Yva).mean():.3f}  F1 "+"  ".join(f"{c} {v:.2f}" for c,v in zip(CLASS_NAMES,F))+f"   ({time.time()-t0:.0f}s)")

In [ ]:
# central slice through one validation sub-cube
net3.eval()
with torch.no_grad(): pv=net3(Xv).argmax(1).cpu().numpy()
zc=patch//2
fig,ax=plt.subplots(1,3,figsize=(15,5))
ax[0].imshow(Xva[0,zc].T,origin='lower',cmap='inferno'); ax[0].set_title('sparse tracers (slice of 3D cube)')
ax[1].imshow(Yva[0,zc].T,origin='lower',cmap=CMAP,norm=NORM); ax[1].set_title('true web (3D)')
im=ax[2].imshow(pv[0,zc].T,origin='lower',cmap=CMAP,norm=NORM); ax[2].set_title('3D U-Net prediction')
cb=plt.colorbar(im,ax=ax[2],ticks=[0,1,2,3],fraction=.046); cb.ax.set_yticklabels(CLASS_NAMES)
plt.tight_layout(); plt.show()

## Next: the real heavy-duty stuff

This whole notebook runs on *toy* fields. The graduation step is real N-body data:

- **Load a real QUIJOTE density field** (`df_m_*.npy`) via Globus → `np.load` it, label it with `tweb`, and feed the same pipeline. Or read a compressed snapshot with `h5py` + `hdf5plugin` and grid it.
- **Scale the 3D net on the GPU** — bigger `N`, more sub-cubes, several realizations — to lift the rare-class (knot) F1.
- **Validate against Pylians' spherical void finder** once it's in the WSL env, as an independent check on what the network calls a void.